In [17]:
import json
import os
import kagglehub
import importlib
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import pandas as pd
import numpy as np
import gc
from src.utils import load_indices_from_jsonl
from IPython.display import JSON
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from functools import partial
from src.jsonl_dataset import JsonLDataset
from transformers import AutoTokenizer, TrainingArguments, EvalPrediction
from adapters import AutoAdapterModel, AdapterTrainer
from adapters.composition import Parallel  # <--- Crucial for parallel execution

In [15]:
path = kagglehub.dataset_download("Cornell-University/arxiv/versions/272")
ds_path = os.path.join(path, 'arxiv-metadata-oai-snapshot.json')
master_ds = JsonLDataset(ds_path)
augmented_ds = JsonLDataset("resources/augmented_index.jsonl")

Indexing dataset at C:\Users\kerem\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\272\arxiv-metadata-oai-snapshot.json... (this may take a minute)
Indexed 2951540 entries.
Indexing dataset at resources/augmented_index.jsonl... (this may take a minute)
Indexed 2951540 entries.


In [18]:
train_indices = load_indices_from_jsonl("resources/train_indicies_parent_categories.jsonl", flatten=True, shuffle=True, seed=42)
test_indices = load_indices_from_jsonl("resources/test_indices_parent_categories.jsonl", flatten=True, shuffle=True, seed=42)

In [117]:
class MultiAdapterInferencePipeline:
    def __init__(
        self, 
        model_name: str = "allenai/specter2_base",
        adapter_configs: list = None, # List of dicts: {"path": "...", "name": "..."}
        mlb = None, 
        device: str = None
    ):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.mlb = mlb
        self.adapter_names = [cfg["name"] for cfg in adapter_configs]

        print(f"Initializing Multi-Adapter Pipeline on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoAdapterModel.from_pretrained(model_name)

        # 1. Load all adapters and their respective heads
        for cfg in adapter_configs:
            print(f"  -> Loading {cfg['name']}...")
            self.model.load_adapter(cfg["path"], load_as=cfg["name"])
        
        # 2. Set them to run in Parallel
        # This tells the model: "In every layer, pass the input through all these adapters"
        self.model.set_active_adapters(Parallel(*self.adapter_names))
        
        self.model.to(self.device)
        self.model.eval()

    def predict(self, items: list, threshold: float = 0.5, weights: list = None):
        """
        Args:
            items: List of dicts, e.g., [{"title": "...", "abstract": "..."}]
            threshold: Sigmoid threshold for multi-label classification.
            weights: Optional list of floats to weight adapter outputs.
        """
        # 1. Replicate Collator text construction
        # We use the same formatting: Title + [SEP] + Abstract
        texts = [
            f"{item['title']}{self.tokenizer.sep_token}{item.get('abstract', '')}" 
            for item in items
        ]
    
        # 2. Tokenization
        inputs = self.tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(self.device)
        
        # Handle default weighting
        if weights is None:
            weights = [1.0 / len(self.adapter_names)] * len(self.adapter_names)
    
        # 3. Model Inference
        with torch.no_grad():
            # Passing inputs through the Parallel adapter setup
            outputs = self.model(**inputs)
            
            # In Parallel mode, outputs is a list of AdapterOutput objects
            # We extract logits from each specific adapter head
            all_probs = []
            for output in outputs:
                logits = output.logits
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
            
            # 4. Weighted Aggregation
            # We ensemble the probabilities from all 3+ adapters
            final_probs = np.zeros_like(all_probs[0])
            for p, w in zip(all_probs, weights):
                final_probs += (p * w)
    
        # 5. Mapping back to Labels
        results = []
        for i, prob_dist in enumerate(final_probs):
            preds = (prob_dist > threshold).astype(int)
            
            # Use inverse_transform to get the original class names
            human_labels = []
            if self.mlb:
                human_labels = self.mlb.inverse_transform(np.expand_dims(preds, axis=0))[0]
                
            results.append({
                "logits": all_probs,
                "probabilities": prob_dist.tolist(),
                "labels": list(human_labels),
                "labels_mlb": self.mlb.transform([list(human_labels)])
            })
    
        return results

In [118]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Define your Fixed Taxonomy
TARGET_PARENT_CLASSES = [
    'Physics', 
    'Mathematics', 
    'Computer Science', 
    'Quantitative Biology', 
    'Statistics', 
    'Quantitative Finance', 
    'Economics', 
    'Electrical Engineering and Systems Science'
]

def create_mlb(target_classes):
    mlb = MultiLabelBinarizer(classes=target_classes)
    mlb.fit([target_classes])
    return mlb

mlb_parent = create_mlb(TARGET_PARENT_CLASSES)

configs = [
    {"path": "./resources/parent_categories_adapter/", "name": "arxiv_parent_categories_classifier"},
    {"path": "./resources/parent_categories_adapter_bucket2/", "name": "arxiv_parent_categories_classifier_bucket2"}
]

parent_inference_pipeline = MultiAdapterInferencePipeline(adapter_configs=configs, mlb=mlb_parent)

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading arxiv_parent_categories_classifier...
  -> Loading arxiv_parent_categories_classifier_bucket2...


There are adapters available but none are activated for the forward pass.


### Manual validation of the predictive quality

In [121]:
from sklearn.metrics import f1_score

In [165]:
rand_idx = random.choice(test_indices)
rand_sample = master_ds[rand_idx]
res = parent_inference_pipeline.predict([rand_sample])
print(f"Predicted: {res[0]['labels']}")
print(f"Actual   : {augmented_ds[rand_idx][2]}")
print(f"Score:   : {f1_score(res[0]['labels_mlb'], mlb_parent.transform([augmented_ds[rand_idx][2]]), average="micro")}")

Predicted: ['Physics']
Actual   : ['Physics']
Score:   : 1.0


In [144]:
res[0]['labels_mlb']

array([[0, 0, 0, 1, 0, 0, 0, 0]])

In [146]:
mlb_parent.transform([augmented_ds[rand_idx][2]])

array([[1, 0, 0, 1, 0, 0, 0, 0]])

In [149]:
f1_score([[1, 0, 0, 1, 0, 0, 0, 0]], [[1, 0, 0, 1, 0, 0, 0, 0]], average="micro")

1.0

In [120]:
res[0]['labels_mlb']

array([[0, 0, 1, 0, 0, 1, 0, 0]])

In [112]:
mlb_parent.transform([res[0]['labels']])

array([[0, 0, 0, 0, 1, 0, 0, 0]])

In [111]:
mlb_parent.transform([augmented_ds[rand_idx][2]])

array([[0, 0, 0, 0, 1, 0, 0, 0]])

### Manual validation of logits aggregation

In [60]:
res[0]['logits'][0][0].tolist()

[0.02983766235411167,
 0.004058793652802706,
 0.9026868343353271,
 0.7659959197044373,
 0.029498063027858734,
 0.0003198585473001003,
 0.0004093350435141474,
 0.7059654593467712]

In [61]:
res[0]['logits'][1][0].tolist()

[0.02192239835858345,
 0.0069285291247069836,
 0.9079641103744507,
 0.6884737610816956,
 0.041807886213064194,
 0.0008135340758599341,
 0.00047387508675456047,
 0.7104717493057251]

In [65]:
((res[0]['logits'][0][0] + res[0]['logits'][1][0]) / 2).tolist()

[0.025880031287670135,
 0.005493661388754845,
 0.9053254723548889,
 0.7272348403930664,
 0.035652972757816315,
 0.0005666962824761868,
 0.0004416050505824387,
 0.7082185745239258]

In [66]:
res[0]['probabilities']

[0.025880031287670135,
 0.005493661388754845,
 0.9053254723548889,
 0.7272348403930664,
 0.035652972757816315,
 0.0005666962824761868,
 0.0004416050505824387,
 0.7082185745239258]

### Manual validation of outputs